In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from scipy.fft import fft, fftfreq
from scipy import signal
from scipy.signal import savgol_filter
from scipy.stats import spearmanr, linregress

In [ ]:
# Util functions

def plot_emg(time_data,emg_data,col_name):
    plt.figure(figsize=(10, 6))

    # Plot for raw data
    #plt.subplot(1, 2, 1)  # 1 row, 2 columns, first subplot
    plt.plot(time_data, emg_data, label="EMG Data", color='blue')
    plt.xlabel("Time (s)")
    plt.ylabel("EMG (V)")
    plt.title(str(col_name))
    plt.grid(True)

    # Display the plot

    plt.tight_layout()
    plt.show()


def plot_emg_wlabels(x, y, name=None, label='Frequencies', gt_time=None):
    plt.figure(figsize=(12, 6))
    plt.plot(x, y)
    plt.title(str(label), fontsize=16) 
    plt.xlabel('Time (s)', fontsize=14)  
    plt.ylabel('Frequency (Hz)', fontsize=14)  
    plt.grid(True)
    plt.axvline(x=gt_time[0], color='black', linestyle='--', label='Non-Fatigue')
    plt.axvline(x=gt_time[1], color='black', linestyle='--', label='Transition-to-Fatigue')
    
    if name is not None:
        plt.savefig(name)
        print("saved at", name)
    plt.close()
    

def mean_frequency(power_spectrum, frequencies):
    """
    Calculate the mean frequency from the power spectrum.
    Args:
    - power_spectrum: Power spectrum from FFT.
    - frequencies: Corresponding frequency bins.
    
    Returns:
    - Mean frequency.
    """

    if len(power_spectrum) == 0 or np.sum(power_spectrum) == 0:
        print("-------- power spectrum is empty --------")
        return None  
    

    weighted_sum = np.sum(frequencies * power_spectrum)
    total_power = np.sum(power_spectrum)
    
    if total_power == 0:
        print( "------------ Total Power is zero --------- ")
        return None
    
    mean_freq = weighted_sum / total_power
    
    return mean_freq


def median_frequency(power_spectrum, frequencies):
    """
    Calculate the median frequency from the power spectrum.
    Args:
    - power_spectrum: Power spectrum from FFT.
    - frequencies: Corresponding frequency bins.
    
    Returns:
    - Median frequency.
    """
    cumulative_power = np.cumsum(power_spectrum)
    total_power = cumulative_power[-1] #last element retrives the total power
    
    # Find the frequency where the cumulative power reaches 50% of the total power
    median_idx = np.where(cumulative_power >= (total_power / 2))[0][0]

    return frequencies[median_idx]


def calculate_median_frequencies(signal, time, sampling_rate, window_size_seconds, step_size_seconds):
    """
    Calculate the median frequency using a sliding window approach.
    Args:
    - signal: The raw signal values.
    - sampling_rate: The sampling rate of the signal.
    - window_size_seconds: Duration of each window in seconds.
    - step_size_seconds: Step size for sliding the window in seconds.
    
    Returns:
    - List of median frequencies for each window.
    """

    window_size = int(window_size_seconds * sampling_rate)  # Number of samples in one window
    step_size = int(step_size_seconds * sampling_rate)      # Number of samples to slide the window
    num_windows = int((len(signal) - window_size) // step_size + 1)
    
    median_frequencies = []
    mean_frequencies = []
    window_time_center = []

    for i in range(num_windows):

        # Get the current window
        start_idx = i * step_size
        end_idx = start_idx + window_size
        window_signal = signal[start_idx:end_idx]
        window_time = time[start_idx:end_idx]

        # Calculate the center time for the current window
        center_time = window_time[int(len(window_time) // 2)]
        window_time_center.append(center_time)

        signal_mean = np.mean(signal)
        #plot(window_time,window_signal, name=filename, label='raw sEMG windows')

        # Perform FFT on the windowed signal
        fft_result = fft(window_signal-signal_mean)
        frequencies = fftfreq(len(window_signal), d=1/sampling_rate)
        power_spectrum = np.abs(fft_result)**2  # Power spectrum

        # Only consider positive frequencies (first half of FFT result)
        positive_frequencies = frequencies[:len(frequencies)//2]
        positive_power_spectrum = power_spectrum[:len(power_spectrum)//2]  

        # Calculate median frequency for this window
        med_freq = median_frequency(positive_power_spectrum, positive_frequencies)
        mean_freq = mean_frequency(positive_power_spectrum, positive_frequencies)
        median_frequencies.append(med_freq)
        mean_frequencies.append(mean_freq)


    return window_time_center, median_frequencies, mean_frequencies


# Function to normalize time by maximum duration
def normalize_time_to_max(raw_time, max_time):
    if raw_time[-1] < max_time:
        # Pad with NaNs to match the max_time
        padding = np.nan * np.ones(int(max_time - len(raw_time)))
        return np.concatenate((raw_time, padding))
    else:
        # Truncate if longer
        return raw_time
    

def plot_overlapping_data(time_data, muscle_data, metric, muscle_name):
    plt.figure(figsize=(20, 14))

    cmap = plt.get_cmap('tab20')  
    colors = [cmap(i) for i in range(13)]  

    line_styles = ['-', '--']  
    line_styles = line_styles * 7  # Repeat to match the number of subjects
    
    # Loop through subjects
    for subject_number in range(1, 14):  # Loop through all subjects (1 to 13)
        subject_key = f'subject_{subject_number}'
        
        for trial_index in range(len(time_data[subject_key])):  # Loop through trials for each subject
                    time = time_data[subject_key][trial_index]
                    data = muscle_data[subject_key][trial_index]
                    if len(time) == len(data):  # Ensure that time and data have matching lengths

                        smoothed_data = savgol_filter(data, window_length=8, polyorder=3)  # Adjust window_length and polyorder for best results

                        # Plot smoothed trend
                        plt.plot(time, smoothed_data, label=f'Subject {subject_number} - {muscle_name}', 
                                 color=colors[subject_number - 1], linestyle=line_styles[subject_number % len(line_styles)])
                    else:
                        print(f"Time and data lengths mismatch for subject {subject_number}, {metric}")

    # Set plot titles and labels once after all subjects are plotted
    #plt.title(f'{muscle_name} ', fontsize=18)
    plt.xlabel('Time (s)', fontsize=18)
    plt.ylabel(f'{metric} (Hz)', fontsize=18)
    plt.legend(loc='best', fontsize=14)  # Move legend outside the plot for clarity
    plt.grid(True)

    # Save the plot
    name = f"{metric}_{muscle_name.replace(' ', '_')}.png"
    filename = os.path.join(dir, name)
    plt.savefig(filename)
    plt.show()
    plt.close()
    print(f"Saved plot at {filename}")

    #associate muscle with trial
def get_prime_mover(trial):
    mapping = {1:1,2:0, # anterior
               3:3,4:2, # posterior
               5:0,6:1, # biceps
               7:2,8:3, # medius
               9:1,10:0, # anterior
               11:3,12:2} # posterior
    return mapping.get(trial,None)

def extract_trial_summary(time_arr, mnf_arr, mdf_arr, subject_number, trial, muscle_name, baseline_n=3):
    time_arr = np.array(time_arr)
    mnf_arr = np.array(mnf_arr)
    mdf_arr = np.array(mdf_arr)
    
    row = {
        'subject': subject_number,
        'trial': trial,
        'muscle': muscle_name,
        'duration_s': time_arr[-1] - time_arr[0],
        'n_windows': len(time_arr),
    }
    
    for name, arr in [('MNF', mnf_arr), ('MDF', mdf_arr)]:
        baseline = np.mean(arr[:baseline_n])
        final = np.mean(arr[-baseline_n:])
        rho, pval = spearmanr(time_arr, arr)
        slope, intercept, r, p, se = linregress(time_arr, arr)
        
        row[f'{name}_baseline'] = baseline
        row[f'{name}_final'] = final
        row[f'{name}_pct_change'] = (final - baseline) / baseline * 100
        row[f'{name}_spearman_rho'] = rho
        row[f'{name}_spearman_p'] = pval
        row[f'{name}_slope'] = slope
        row[f'{name}_r2'] = r**2
        row[f'{name}_cv'] = np.std(arr) / np.mean(arr)
    
    return row

def total_power(power_spectrum):
    return np.sum(power_spectrum)

def rms_windowed(signal, sampling_rate, window_size_seconds, step_size_seconds):
    window_size = int(window_size_seconds * sampling_rate)
    step_size = int(step_size_seconds * sampling_rate)
    num_windows = int((len(signal) - window_size) // step_size + 1)
    
    rms_values = []
    for i in range(num_windows):
        start = i * step_size
        end = start + window_size
        window = signal[start:end]
        rms_values.append(np.sqrt(np.mean(window**2)))
    return rms_values

def extract_gt_features(gt_times, total_duration):
    non_fatigue_end, transition_end = gt_times
    return {
        'gt_non_fatigue_duration': non_fatigue_end,
        'gt_transition_duration': transition_end - non_fatigue_end,
        'gt_non_fatigue_pct': non_fatigue_end / total_duration * 100,
        'gt_transition_pct': (transition_end - non_fatigue_end) / total_duration * 100,
    }

# Trial-to-muscle correspondence dictionary
trial_to_muscle = {
    1: "R DELTOID ANTERIOR",
    2: "L DELTOID ANTERIOR",
    3: "R DELTOID POSTERIOR",
    4: "L DELTOID POSTERIOR",
    5: "R BICEPS BRACHII",
    6: "L BICEPS BRACHII",
    7: "R DELTOID MEDIUS",
    8: "L DELTOID MEDIUS",
    9: "R DELTOID ANTERIOR C",
    10: "L DELTOID ANTERIOR C",
    11: "R DELTOID POSTERIOR C",
    12: "L DELTOID POSTERIOR C",
}

                

In [ ]:
dir = r'D:\smilecow\HUST\Nam_3\nckh\dataset\sEMG_data'
GT_dir = r'D:\smilecow\HUST\Nam_3\nckh\dataset\self_perceived_fatigue_index'

subject_numbers = range(1, 14)  # Adjust the range as needed

not_filtered = True
X_subjects = []
true_labels_subjects = []
time= []
EMG_index = [1,3,5,7]
time_index = [0,2,4,6]
sample_freq = 1259 #Hz




# Initialize a dictionary to store the data for each muscle
# Initialize a dictionary to store the MNF/MDF data for each muscle, subject, and trial
muscle_data_MNF = {f'muscle_{i}': {f'subject_{j}': [] for j in range(1, 14)} for i in range(1, 14)}
muscle_data_MDF = {f'muscle_{i}': {f'subject_{j}': [] for j in range(1, 14)} for i in range(1, 14)}
time_dic = {f'muscle_{i}': {f'subject_{j}': [] for j in range(1, 14)} for i in range(1, 14)}

# Step 1: Find the maximum time across all subjects and trials
max_time = 0

# Loop through each subject
for subject_number in subject_numbers:
    emg_directory = os.path.join(dir, f'subject_{subject_number}')
    #GT_directory = os.path.join(dir, f'Subject_{subject_number}')
    
    emg_files = sorted([f for f in os.listdir(emg_directory) if f.endswith(".csv")], key=lambda x: int(re.findall(r'\d+', x)[-1]))
    #GT_files = sorted([f for f in os.listdir(GT_directory) if f.endswith(".csv")], key=lambda x: int(re.findall(r'\d+', x)[-1]))

    data_subject = []  # List to store data for current subject
    labels_subject = []  # List to store true labels for current subject

    for emg_file in zip(emg_files):
        emg_path = os.path.join(emg_directory, emg_file[0])

        file = pd.read_csv(emg_path, delimiter=',', header=0)
        file_values = file.values
        columns_names = file.columns
        trial = int(re.findall(r'\d+', emg_file[0])[-1])
    
        # Loop through EMG and time indices
        index = get_prime_mover(trial)
        print("trial", trial, index)
        

        emg_data = file_values[:, EMG_index[index]]
        time_data = file_values[:, time_index[index]]
        col_name = columns_names[EMG_index[index]]
        #plot_emg(time_data, emg_data,col_name)

        if not_filtered:
            fc1, fc2 = 20, 450
            b, a = signal.butter(4, [fc1, fc2], btype='bandpass', fs=sample_freq)
            filtered_emg = signal.filtfilt(b, a, emg_data)
            emg_data = filtered_emg


        # Define window and step size
        window_size_seconds = 4  # 4-second window
        step_size_seconds = 2    # 50% overlap
        #Overlapping windows help reduce abrupt changes between segments and provide a more continuous analysis of the signal.

        # Calculate median frequencies
        window_time_centers, median_frequencies, mean_frequencies = calculate_median_frequencies(emg_data,time_data, sample_freq, window_size_seconds, step_size_seconds )

        ## ---- Single emg  plots --- ##
        #plot_emg(window_time_centers,median_frequencies,("MDF" + col_name))
        #plot_emg(window_time_centers,mean_frequencies,("MNF" + col_name))

        max_time = max(max_time, window_time_centers[-1])  # Update the maximum time
        

        # Store the data by muscle

        muscle_data_MNF[f'muscle_{trial}'][f'subject_{subject_number}'].append(mean_frequencies)
        muscle_data_MDF[f'muscle_{trial}'][f'subject_{subject_number}'].append(median_frequencies)
        time_dic[f'muscle_{trial}'][f'subject_{subject_number}'].append(window_time_centers)

for trial in range(1, 13):
    # Get the corresponding muscle name from the trial index
    muscle_name = trial_to_muscle.get(trial)
    print(f"Plotting data for {muscle_name}")

    # Check if there is any data to plot
    if time_dic[f'muscle_{trial}'] and muscle_data_MNF[f'muscle_{trial}']:
        # Plot MNF
        plot_overlapping_data(time_dic[f'muscle_{trial}'], muscle_data_MNF[f'muscle_{trial}'], 
                              'Mean Frequency', f'{muscle_name}')

    if time_dic[f'muscle_{trial}'] and muscle_data_MDF[f'muscle_{trial}']:
        # Plot MDF
        plot_overlapping_data(time_dic[f'muscle_{trial}'], muscle_data_MDF[f'muscle_{trial}'], 
                              'Median Frequency', f'{muscle_name}')
    else:
        print(f"No data to plot for muscle {muscle_name}")






In [ ]:
## Frequency analysis  - MDF and MNF
# PLOT a graph with the MDF values of 1 cycle
# velocity was 30 bpm, meaning that there was a beat every 2 seconds. 
#therefore, the cycle was 2 seconds and the window size should also be 

from scipy import signal

base_directory = r'D:\smilecow\HUST\Nam_3\nckh\dataset'

EMG_index = [1,3,5,7]
time_index = [0,2,4,6]
sample_freq = 1259 #Hz

for subject_number in range(1, 14): 

  path_directory = os.path.join(base_directory, 'sEMG_data', f'subject_{subject_number}')
  label_directory = os.path.join(base_directory, 'self_perceived_fatigue_index', f'subject_{subject_number}')
  global out_directory
  out_directory = os.path.join(base_directory, '','frequency_analysis', 'per_trial',f'subject_{subject_number}')

  isExist = os.path.exists(out_directory)
  if not isExist:


    os.makedirs(out_directory)


  files = [file_name for file_name in os.listdir(path_directory) if file_name.endswith("csv")]
  files = sorted(files, key=lambda x: int(re.findall(r'\d+', x)[-1]))

  label_files = [file_name for file_name in os.listdir(label_directory) if file_name.endswith("csv")]
  label_files = sorted(label_files, key=lambda x: int(re.findall(r'\d+', x)[-1]))
  
  for file_raw, file_down in zip(files, label_files):
        # Read the CSV file
        
        raw_files_path = os.path.join(path_directory, file_raw)
        label_files_path = os.path.join(label_directory, file_down)

        trial = re.findall(r'\d+', file_raw)
        print("file", file_raw, "trial", int(trial[0]))
        index = get_prime_mover(int(trial[0]))


        raw = pd.read_csv(raw_files_path, delimiter=',', header=0)
        file_values = raw.values
        raw_time =  file_values[:, time_index[index]]
        raw_val =  file_values[:, EMG_index[index]]
        if not_filtered:
          fc1, fc2 = 20, 450
          b, a = signal.butter(4, [fc1, fc2], btype='bandpass', fs=sample_freq)
          filtered_emg = signal.filtfilt(b, a, raw_val)
          raw_val = filtered_emg

        label = pd.read_csv(label_files_path, delimiter=',', header=0)
        label_time = label.iloc[:, 0]  
        label_values = label.iloc[:, 1]

        raw_values = np.array(raw_val)
        if np.isnan(raw_val).any():
          print("NaN values found in the signal. Replacing NaN with zeros.")
    
        if np.isinf(raw_val).any():
            print("Inf values found in the signal. Replacing Inf with zeros.")

        raw_values = np.where(np.isnan(raw_val) | np.isinf(raw_val), 0, raw_val)

      # ----- Segment 0: From the beginning to the last 0 in label_values
        idx_last_0 = label_values[label_values == 0].index[-1]
        segment_0_label_time = label_time[idx_last_0]
        # Find the index in raw_time that corresponds to the nearest time to segment_1_label_time
        nearest_idx = (np.abs(raw_time - segment_0_label_time)).argmin()
        # Nearest time and values in raw_time
        nearest_time = raw_time[nearest_idx]

        # ------- Segment 1: From the the fisrt 1 to the last 1 in label_values
        idx_last_1 = label_values[label_values == 1].index[-1]
        segment_1_label_time = label_time[idx_last_1]
        nearest_idx_1 = (np.abs(raw_time - segment_1_label_time)).argmin()
        nearest_time_1 = raw_time[nearest_idx_1]

        gt_times = [nearest_time, nearest_time_1]
  
        # Define window and step size FOR MDF AND MNF ANALYSIS
        window_size_seconds = 4  # 4-second window
        step_size_seconds = 2    # 4-second step (50% overlap) 
        #Overlapping windows help reduce abrupt changes between segments and provide a more continuous analysis of the signal.

        # Calculate median frequencies
        window_time_centers, median_frequencies,mean_frequencies = calculate_median_frequencies(raw_values,raw_time, sample_freq, window_size_seconds, step_size_seconds )

        print(f"Median Frequencies for {file_raw}: {median_frequencies}")
        freq_MDF_name = os.path.join(out_directory,f'MDF_{(file_raw.replace(".csv",".png"))}')
        #id = np.arange(0,len(median_frequencies),1)
        plot_emg_wlabels(window_time_centers,median_frequencies,name=freq_MDF_name, label="median frequencies", gt_time=gt_times)


        #n = np.arange(0,len(mean_frequencies),1)
        freq_name = os.path.join(out_directory,f'MNF_{(file_raw.replace(".csv",".png"))}')
        plot_emg_wlabels(window_time_centers,mean_frequencies, name = freq_name, label='mean frequencies', gt_time=gt_times)
        print ("---")

In [ ]:
# ============================================================
# PHASE A: Bảng tổng hợp cấp độ Trial (Trial-Level Summary)
# PHASE D: Z-score Outlier detection, aggregate theo cơ/subject
# Phụ thuộc: phải chạy cell xử lý chính (tính MNF/MDF) trước.
# ============================================================
import matplotlib.pyplot as plt

summary_rows = []

for trial in range(1, 13):
    muscle_name = trial_to_muscle.get(trial)
    for subject_number in range(1, 14):
        subject_key = f'subject_{subject_number}'
        muscle_key  = f'muscle_{trial}'

        time_list = time_dic[muscle_key][subject_key]
        mnf_list  = muscle_data_MNF[muscle_key][subject_key]
        mdf_list  = muscle_data_MDF[muscle_key][subject_key]

        if not time_list or not mnf_list:
            continue

        for ti in range(len(time_list)):
            t   = np.array(time_list[ti])
            mnf = np.array(mnf_list[ti])
            mdf = np.array(mdf_list[ti])
            if len(t) < 3:
                continue
            row = extract_trial_summary(t, mnf, mdf, subject_number, trial, muscle_name, baseline_n=3)
            row['movement_type'] = 'complex' if ' C' in muscle_name else 'uni-articular'
            summary_rows.append(row)

trial_df = pd.DataFrame(summary_rows)
print(f"[Phase A] {len(trial_df)} trial rows built.")

# ── Z-score outlier detection (so sánh trong cùng 1 nhóm cơ) ──
for metric in ['MNF_spearman_rho', 'MDF_spearman_rho', 'MNF_cv', 'MDF_cv',
               'MNF_baseline', 'MDF_baseline']:
    trial_df[f'{metric}_zscore'] = trial_df.groupby('muscle')[metric].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-9)
    )

# Cờ outlier: Spearman >= 0 (đi ngược lý thuyết) HOẶC z-score cực đoan
trial_df['is_outlier'] = (
    (trial_df['MNF_spearman_rho'] >= 0) |
    (trial_df['MNF_spearman_rho_zscore'].abs() > 2.5)
)

n_out = trial_df['is_outlier'].sum()
print(f"[Phase D] Outlier: {n_out}/{len(trial_df)} trial ({n_out/len(trial_df)*100:.1f}%)")
print("\nTrial bị gắn cờ outlier:")
print(trial_df[trial_df['is_outlier']][
    ['subject','muscle','movement_type','MNF_baseline',
     'MNF_pct_change','MNF_spearman_rho','MNF_r2']
].to_string(index=False))

# ── Aggregate theo nhóm cơ ──
print("\n─── Aggregate theo Nhóm cơ ───")
muscle_agg = trial_df.groupby('muscle').agg(
    MNF_rho_mean  =('MNF_spearman_rho','mean'),
    MNF_rho_std   =('MNF_spearman_rho','std'),
    MNF_r2_mean   =('MNF_r2','mean'),
    MNF_slope_mean=('MNF_slope','mean'),
    MNF_cv_mean   =('MNF_cv','mean'),
    MNF_base_mean =('MNF_baseline','mean'),
    MNF_base_std  =('MNF_baseline','std'),
    dur_mean       =('duration_s','mean'),
    dur_min        =('duration_s','min'),
    dur_max        =('duration_s','max'),
    n_outliers     =('is_outlier','sum'),
).round(3)
print(muscle_agg.to_string())

# ── Aggregate theo Subject ──
print("\n─── Aggregate theo Subject ───")
subj_agg = trial_df.groupby('subject').agg(
    MNF_rho_mean =('MNF_spearman_rho','mean'),
    MNF_rho_std  =('MNF_spearman_rho','std'),
    MNF_cv_mean  =('MNF_cv','mean'),
    dur_mean      =('duration_s','mean'),
    n_outliers    =('is_outlier','sum'),
).round(3)
print(subj_agg.to_string())

# ── Lưu CSV ──
out_csv = os.path.join(dir, 'trial_summary_with_flags.csv')
trial_df.to_csv(out_csv, index=False)
print(f"\n[Phase A/D] Saved → {out_csv}")


In [ ]:
import matplotlib
print(matplotlib.__version__)

In [ ]:
import matplotlib.pyplot as plt

muscles_sorted = (trial_df.groupby('muscle')['MNF_spearman_rho']
                  .mean().sort_values().index.tolist())
short = [m.replace('DELTOID','DEL').replace('BICEPS BRACHII','BICEPS') for m in muscles_sorted]

# 1) Spearman correlation
fig, ax = plt.subplots(figsize=(16, 8))
ax.boxplot(data_rho)
ax.set_xticklabels(short, rotation=55)
ax.axhline(0, color='red', linestyle='--', lw=1.5)
ax.set_title('Spearman Correlation of MNF by Muscle Group', fontsize=14, fontweight='bold')
ax.set_xlabel('Muscle Group')
ax.set_ylabel('Spearman ρ')
plt.tight_layout()
fig_path = os.path.join(dir, 'quality_boxplot_spearman_rho.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {fig_path}")

# 2) Coefficient of determination
fig, ax = plt.subplots(figsize=(16, 8))
ax.boxplot(data_r2)
ax.set_xticklabels(short, rotation=55)
ax.axhline(0.2, color='orange', linestyle='--', lw=1.5)
ax.set_title('Coefficient of Determination (R²) of MNF by Muscle Group', fontsize=14, fontweight='bold')
ax.set_xlabel('Muscle Group')
ax.set_ylabel('R²')
plt.tight_layout()
fig_path = os.path.join(dir, 'quality_boxplot_r2.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {fig_path}")

# 3) Baseline mean frequency
fig, ax = plt.subplots(figsize=(16, 8))
ax.boxplot(data_base)
ax.set_xticklabels(short, rotation=55)
ax.set_title('Baseline Mean Frequency (MNF) by Muscle Group', fontsize=14, fontweight='bold')
ax.set_xlabel('Muscle Group')
ax.set_ylabel('Baseline MNF (Hz)')
plt.tight_layout()
fig_path = os.path.join(dir, 'quality_boxplot_mnf_baseline.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {fig_path}")

# 4) Duration by movement type
fig, ax = plt.subplots(figsize=(12, 8))
ax.boxplot([uni, comp])
ax.set_xticklabels(['Uni-articular', 'Complex'], rotation=55)
ax.set_title('Trial Duration by Movement Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Movement Type')
ax.set_ylabel('Duration (s)')
plt.tight_layout()
fig_path = os.path.join(dir, 'quality_boxplot_duration_by_movement.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {fig_path}")

# ── Scatter: duration vs MNF slope (người mỏi nhanh có slope dốc hơn?) ──
fig2, ax = plt.subplots(figsize=(10, 6))
for mt, color in [('uni-articular','steelblue'), ('complex','tomato')]:
    sub = trial_df[trial_df['movement_type']==mt]
    ax.scatter(sub['duration_s'], sub['MNF_slope'], alpha=0.6, c=color, label=mt, s=40)
ax.axhline(0, color='black', linestyle='--', lw=0.8)
ax.set_xlabel('Trial Duration (s)')
ax.set_ylabel('MNF Slope (Hz/s)')
ax.set_title('Thời lượng vs Tốc độ suy giảm MNF')
ax.legend()
plt.tight_layout()
fig2_path = os.path.join(dir, 'duration_vs_slope.png')
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {fig2_path}")


In [ ]:
# ============================================================
# PHASE C: Khai thác Ground Truth (self_perceived_fatigue_index)
# Câu hỏi kiểm chứng: "sEMG có phát hiện mỏi SỚM HƠN
# cảm nhận chủ quan của subject không?"
# ============================================================

gt_rows = []

for subject_number in subject_numbers:
    path_dir  = os.path.join(dir, 'sEMG_data',
                             f'subject_{subject_number}')
    label_dir = os.path.join(dir, 'self_perceived_fatigue_index',
                             f'subject_{subject_number}')
    if not os.path.exists(path_dir) or not os.path.exists(label_dir):
        print(f"  Skip subject_{subject_number}: thư mục không tồn tại")
        continue

    emg_files = sorted([f for f in os.listdir(path_dir) if f.endswith(".csv")],
                       key=lambda x: int(re.findall(r'\d+', x)[-1]))
    lbl_files = sorted([f for f in os.listdir(label_dir) if f.endswith(".csv")],
                       key=lambda x: int(re.findall(r'\d+', x)[-1]))

    for file_emg, file_lbl in zip(emg_files, lbl_files):
        trial = int(re.findall(r'\d+', file_emg)[0])
        muscle_name = trial_to_muscle.get(trial)
        if muscle_name is None:
            continue

        index = get_prime_mover(trial)

        raw   = pd.read_csv(os.path.join(path_dir, file_emg))
        vals  = raw.values
        raw_time = vals[:, time_index[index]]
        raw_emg  = vals[:, EMG_index[index]]

        # Lọc bandpass
        fc1, fc2 = 20, 450
        b_f, a_f = signal.butter(4, [fc1, fc2], btype='bandpass', fs=sample_freq)
        clean_emg = np.nan_to_num(raw_emg, nan=0.0, posinf=0.0, neginf=0.0)
        filtered  = signal.filtfilt(b_f, a_f, clean_emg)

        # Đọc ground truth labels
        lbl = pd.read_csv(os.path.join(label_dir, file_lbl))
        lbl_time = lbl.iloc[:, 0]
        lbl_vals = lbl.iloc[:, 1]

        try:
            non_fatigue_end = lbl_time[lbl_vals[lbl_vals == 0].index[-1]]
            transition_end  = lbl_time[lbl_vals[lbl_vals == 1].index[-1]]
        except (IndexError, KeyError):
            print(f"  Skip subj={subject_number} trial={trial}: GT label lỗi")
            continue

        total_dur = float(raw_time[-1] - raw_time[0])
        gt_feats  = extract_gt_features([non_fatigue_end, transition_end], total_dur)

        # Tính MNF theo sliding window
        t_centers, _, mnf_vals = calculate_median_frequencies(
            filtered, raw_time, sample_freq, 4, 2)
        if len(t_centers) < 3:
            continue

        t_arr   = np.array(t_centers)
        mnf_arr = np.array(mnf_vals)

        base_n = min(3, len(mnf_arr))
        mnf_baseline = np.mean(mnf_arr[:base_n])

        # MNF tại thời điểm GT non_fatigue_end
        idx_gt     = min(np.searchsorted(t_arr, non_fatigue_end), len(mnf_arr)-1)
        mnf_at_gt  = float(mnf_arr[idx_gt])
        mnf_drop_pct = (mnf_at_gt - mnf_baseline) / mnf_baseline * 100                        if mnf_baseline != 0 else float('nan')

        gt_rows.append({
            'subject'                  : subject_number,
            'trial'                    : trial,
            'muscle'                   : muscle_name,
            'movement_type'            : 'complex' if ' C' in muscle_name else 'uni-articular',
            'total_duration_s'         : total_dur,
            'MNF_baseline_Hz'          : mnf_baseline,
            'MNF_at_gt_nonfatigue_end' : mnf_at_gt,
            'MNF_drop_pct_at_gt'       : mnf_drop_pct,
            **gt_feats,
        })

gt_df = pd.DataFrame(gt_rows)
gt_csv = os.path.join(dir, 'gt_crossref.csv')
gt_df.to_csv(gt_csv, index=False)
print(f"[Phase C] Saved {len(gt_df)} rows → {gt_csv}\n")

print("─── MNF đã giảm bao nhiêu tại thời điểm chủ quan bắt đầu mỏi? ───")
print("(Âm = sEMG thấy mỏi trước cảm nhận; dương = sEMG chậm hơn cảm nhận)")
print(gt_df.groupby('muscle').agg(
    MNF_drop_mean        =('MNF_drop_pct_at_gt','mean'),
    MNF_drop_std         =('MNF_drop_pct_at_gt','std'),
    gt_nonfatigue_pct_mean=('gt_non_fatigue_pct','mean'),
).round(2).to_string())

print("\n─── Thống kê theo subject ───")
print(gt_df.groupby('subject').agg(
    tot_dur_mean        =('total_duration_s','mean'),
    gt_nonfat_pct_mean  =('gt_non_fatigue_pct','mean'),
    MNF_drop_mean       =('MNF_drop_pct_at_gt','mean'),
).round(2).to_string())


In [ ]:
# ============================================================
# PHASE B: Trích đặc trưng bổ sung – Total Power (TP) và RMS
# Tính song song với MNF/MDF trong cùng 1 vòng lặp windowing
# để tránh chạy FFT hai lần.
# ============================================================

def calculate_all_features(sig, time_arr, sampling_rate,
                            window_size_s=4, step_size_s=2):
    """
    Trả về 5 chuỗi: time_centers, MDF, MNF, TP, RMS
    cho mỗi sliding window.
    """
    win = int(window_size_s * sampling_rate)
    stp = int(step_size_s  * sampling_rate)
    n_win = int((len(sig) - win) // stp + 1)

    t_centers  = []
    mdf_list   = []
    mnf_list   = []
    tp_list    = []
    rms_list   = []

    sig_mean = np.mean(sig)

    for i in range(n_win):
        s, e = i * stp, i * stp + win
        w_sig  = sig[s:e]
        w_time = time_arr[s:e]

        # 1) Time-centre
        t_centers.append(w_time[len(w_time) // 2])

        # 2) RMS (time-domain)
        rms_list.append(np.sqrt(np.mean(w_sig ** 2)))

        # 3) FFT (frequency-domain)
        fft_res = fft(w_sig - sig_mean)
        freqs   = fftfreq(len(w_sig), d=1.0 / sampling_rate)
        p_spec  = np.abs(fft_res) ** 2

        pos_f = freqs[:len(freqs) // 2]
        pos_p = p_spec[:len(p_spec) // 2]

        # 4) Total Power
        tp_list.append(total_power(pos_p))

        # 5) MDF
        mdf_list.append(median_frequency(pos_p, pos_f))

        # 6) MNF
        mnf_list.append(mean_frequency(pos_p, pos_f))

    return t_centers, mdf_list, mnf_list, tp_list, rms_list


# Demo: chạy thử trên subject 1, trial 5 (R Biceps)
demo_subj  = 1
demo_trial = 5
demo_index = get_prime_mover(demo_trial)
demo_path  = os.path.join(dir, 'sEMG_data',
                          f'subject_{demo_subj}',
                          f'trial_{demo_trial}.csv')

if os.path.exists(demo_path):
    df_demo  = pd.read_csv(demo_path)
    vals_d   = df_demo.values
    raw_t    = vals_d[:, time_index[demo_index]]
    raw_e    = vals_d[:, EMG_index[demo_index]]
    raw_e    = np.nan_to_num(raw_e, nan=0.0)

    fc1, fc2 = 20, 450
    b_d, a_d = signal.butter(4, [fc1, fc2], btype='bandpass', fs=sample_freq)
    filt_e   = signal.filtfilt(b_d, a_d, raw_e)

    t_c, mdf_d, mnf_d, tp_d, rms_d = calculate_all_features(
        filt_e, raw_t, sample_freq, 4, 2)

    demo_feat_df = pd.DataFrame({
        'time_center': t_c,
        'MDF_Hz'     : mdf_d,
        'MNF_Hz'     : mnf_d,
        'TotalPower' : tp_d,
        'RMS'        : rms_d,
    })
    print(f"[Phase B] Demo subject={demo_subj} trial={demo_trial} "
          f"({trial_to_muscle[demo_trial]}):")
    print(demo_feat_df.head(10).to_string(index=False))

    # Vẽ minh hoạ 4 đặc trưng
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    for ax, col, lbl in zip(axes,
        ['MNF_Hz','MDF_Hz','TotalPower','RMS'],
        ['MNF (Hz)','MDF (Hz)','Total Power','RMS']):
        ax.plot(demo_feat_df['time_center'], demo_feat_df[col])
        ax.set_ylabel(lbl); ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(f'4 Đặc trưng – Subject {demo_subj} '
                 f'Trial {demo_trial} ({trial_to_muscle[demo_trial]})',
                 fontsize=13)
    plt.tight_layout()
    fig_b = os.path.join(dir, f'features_s{demo_subj}_t{demo_trial}.png')
    plt.savefig(fig_b, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {fig_b}")
else:
    print(f"Không tìm thấy: {demo_path}")
